# Validation — `validation`

**Target metric:** `avg_extraneous_tool_rate`

**Repository:** [salma-remyx/openai-agents-python](https://github.com/salma-remyx/openai-agents-python) at commit [`0fa7004141ae`](https://github.com/salma-remyx/openai-agents-python/commit/0fa7004141ae030b3a8d314a98104ff629c88bb9)

**Benchmark:** the command `python tests/mcp/eval_causal_tool_filter_precision.py` named by `suite:` in this repository's `.remyx/validation.yaml` — the measurement is the JSON metrics line it prints.

**Verdict: `pass`** · Remyx run `388c10d1`

The outputs below are the ones that run produced on Remyx compute. Run the cells top to bottom on a machine with a GPU to reproduce them.

In [ ]:
# Parameters (Remyx passes the commit it measures as `ref`)
variant = "feature"
ref = ""
seed = 0

## 1. Environment

On Remyx compute this ran in `python:3.12-slim`; the exact Dockerfile is committed at `.remyx/validations/validation.Dockerfile` — edit it and the next run builds from your version.

A CUDA GPU is required; the published protocol peaks above 22 GB.

In [ ]:
!nvidia-smi -L
import sys, torch
print(f"python {sys.version.split()[0]} · torch {torch.__version__} · cuda {torch.cuda.is_available()}")

## 2. The code under test

Clone the repository and check out exactly the commit that was validated, then install it in editable mode so the harness imports this checkout. When this notebook runs on Remyx compute the checkout already exists at that commit, and this cell only confirms it.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/salma-remyx/openai-agents-python"
COMMIT = ref or "0fa7004141ae030b3a8d314a98104ff629c88bb9"

def _sh(*cmd):
    return subprocess.run(cmd, check=True, text=True, capture_output=True).stdout.strip()

def _at_commit():
    try:
        return os.path.isdir(".git") and _sh("git", "rev-parse", "HEAD").startswith(COMMIT)
    except Exception:
        return False

if not _at_commit():
    if not os.path.isdir("repo"):
        _sh("git", "clone", "--quiet", REPO_URL, "repo")
    os.chdir("repo")
    _sh("git", "fetch", "--quiet", "--depth=1", "origin", COMMIT)
    _sh("git", "checkout", "--quiet", COMMIT)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", "."], check=True)
ROOT = os.getcwd()
print(ROOT)
print(_sh("git", "log", "-1", "--oneline"))

## 3. Credentials

If the benchmark downloads gated models or datasets it needs a Hugging Face token. In Colab, store it as a secret named `HF_TOKEN`; elsewhere set the environment variable.

In [ ]:
import os
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass
print("HF_TOKEN set" if os.environ.get("HF_TOKEN") else "HF_TOKEN not set — gated downloads will fail")

## 4. Confirm the change under test is what is loaded

The commit printed here must match the one checked out above.

In [ ]:
import importlib
print(_sh("git", "rev-parse", "HEAD"))

## 5. Run the benchmark

```sh
python tests/mcp/eval_causal_tool_filter_precision.py
```

Run from the repository root. The last line it prints as `METRICS_JSON={...}` (or as a bare JSON object) is the measurement.

In [ ]:
os.chdir(ROOT)
import subprocess, time
RUN_STARTED = time.time()
COMMAND = "python tests/mcp/eval_causal_tool_filter_precision.py"
proc = subprocess.Popen(COMMAND, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
LOG = []
for line in proc.stdout:
    print(line, end="")
    LOG.append(line)
proc.wait()
print(f"[exit {proc.returncode}]")

===== stdout =====
{"avg_extraneous_tool_rate": 0.0, "necessary_tool_coverage": 1.0}

===== stderr =====

## 6. Read what the benchmark printed

The metrics the criteria are judged against are the fields of the JSON line above. A number here that the verdict did not report is a Remyx fault, not the test's.

In [ ]:
import json

def _printed(lines):
    """The last METRICS_JSON= line the command printed, else its last JSON object line."""
    lines = [l.strip() for l in lines if l.strip()]
    for line in reversed(lines):
        if line.startswith("METRICS_JSON="):
            try:
                return json.loads(line[len("METRICS_JSON="):])
            except ValueError:
                pass
    for line in reversed(lines):
        if line.startswith("{") and line.endswith("}"):
            try:
                return json.loads(line)
            except ValueError:
                pass
    return {}

printed = _printed(LOG)
METRICS = ["avg_extraneous_tool_rate", "necessary_tool_coverage"]
observed = {name: printed.get(name) for name in METRICS}
print(json.dumps(observed, indent=2))

{
  "avg_extraneous_tool_rate": 0.0,
  "necessary_tool_coverage": 1.0
}


## 7. Against the criteria

Thresholds come from `.remyx/validation.yaml`, so a failing measurement reports rather than crashes. `baseline` is the published row this repository already ships for the comparison method.

In [ ]:
CRITERIA = [
    {
        "metric": "avg_extraneous_tool_rate",
        "direction": "<=",
        "threshold": 0.05,
        "baseline": 0.833333
    },
    {
        "metric": "necessary_tool_coverage",
        "direction": ">=",
        "threshold": 0.0,
        "baseline": 1.0
    }
]

print(f"{'metric':<28}{'observed':>16}{'baseline':>16}  criterion")
for c in CRITERIA:
    v = observed.get(c["metric"])
    t = c["threshold"]
    ok = None if v is None or t is None else (v <= t if c["direction"] == "<=" else v >= t)
    mark = "?" if ok is None else ("PASS" if ok else "FAIL")
    fmt = lambda x: (f"{x:.6g}" if isinstance(x, float) else str(x))
    print(f"{c['metric']:<28}{fmt(v):>16}{fmt(c['baseline']):>16}  {c['direction']} {fmt(t)}  {mark}")

metric                              observed        baseline  criterion
avg_extraneous_tool_rate                 0.0        0.833333  <= 0.05  PASS
necessary_tool_coverage                  1.0             1.0  >= 0.0  PASS


## 8. Report

One line, machine-readable — what Remyx records as this run's measurement.

In [ ]:
print(json.dumps(observed))

{"avg_extraneous_tool_rate": 0.0, "necessary_tool_coverage": 1.0}


## 9. What the outcome means

**Verdict: `pass`.** Every change the validation loop made on the way is a separate commit on this branch, named for the run that applied it — revert any of them to undo it.

## Appendix — the criteria file

`.remyx/validation.yaml` as committed:

```yaml
benchmarks:
  - suite: "command:python tests/mcp/eval_causal_tool_filter_precision.py"
    scorer: avg_extraneous_tool_rate
    baseline: main
    metrics:
      - name: avg_extraneous_tool_rate
        role: target
        direction: decrease
        threshold: 0.05
      - name: necessary_tool_coverage
        role: guardrail
        direction: increase
        threshold: 0.0
    policy:
      guardrail_veto: true
```